In [ ]:
# %% [markdown]
# # Spanner Data Agent End-to-End Example
#
# This notebook demonstrates how to create and interact with a Data Agent for Spanner using the Gemini Data Analytics API via HTTP requests in Python.


In [ ]:
# Define helper functions

import json as json_lib
import textwrap
import pandas as pd
from IPython.display import display, HTML
import altair as alt
from pygments import formatters, highlight, lexers
import requests

def is_json(my_str):
    """Checks if a string is valid JSON."""
    try:
        json_lib.loads(my_str)
    except ValueError:
        return False
    return True

def handle_text_response(resp):
    """Handles and prints text responses, wrapping long lines."""
    parts = resp.get('parts', [])
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)

def get_property(data, field_name, default=''):
    """Safely gets a property from a dictionary."""
    return data.get(field_name, default)

def display_schema(data):
    """Displays schema information in a DataFrame."""
    fields = data.get('fields', [])
    df = pd.DataFrame({
        "Column": [get_property(field, 'name') for field in fields],
        "Type": [get_property(field, 'type') for field in fields],
        "Description": [get_property(field, 'description', '-') for field in fields],
        "Mode": [get_property(field, 'mode') for field in fields]
    })
    display(df)

def display_section_title(text):
    """Displays a formatted section title in HTML."""
    display(HTML(f'<h2>{text}</h2>'))

def format_bq_table_ref(table_ref):
    """Formats a BigQuery table reference for display."""
    return f"{table_ref.get('projectId')}.{table_ref.get('datasetId')}.{table_ref.get('tableId')}"

def format_looker_table_ref(table_ref):
    """Formats a Looker table reference for display."""
    return f"lookmlModel: {table_ref.get('lookmlModel')}, explore: {table_ref.get('explore')}, lookerInstanceUri: {table_ref.get('lookerInstanceUri')}"

def display_datasource(datasource):
    """Displays information about a datasource, including its schema."""
    source_name = ''
    if 'studioDatasourceId' in datasource:
        source_name = datasource['studioDatasourceId']
    elif 'lookerExploreReference' in datasource:
        source_name = format_looker_table_ref(datasource['lookerExploreReference'])
    elif 'bigqueryTableReference' in datasource:
        source_name = format_bq_table_ref(datasource['bigqueryTableReference'])
    else:
        source_name = "Unknown Datasource"

    print(source_name)
    if 'schema' in datasource:
        display_schema(datasource['schema'])

def handle_schema_response(resp):
    """Handles responses related to schema resolution."""
    if 'query' in resp:
        print(get_property(resp['query'], 'question'))
    elif 'result' in resp:
        display_section_title('Schema resolved')
        print('Data sources:')
        for datasource in get_property(resp['result'], 'datasources', []):
            display_datasource(datasource)

def handle_data_response(resp):
    """Handles responses containing data or SQL queries."""
    if 'query' in resp:
        query = resp['query']
        display_section_title('Retrieval query')
        print(f"Query name: {get_property(query, 'name')}")
        if 'question' in query:
            print(f"Question: {get_property(query, 'question')}")
        if 'datasources' in query:
            print('Data sources:')
            for datasource in get_property(query, 'datasources', []):
                display_datasource(datasource)
    elif 'generatedSql' in resp:
        display_section_title('SQL generated')
        print(resp['generatedSql'])
    elif 'result' in resp:
        display_section_title('Data retrieved')
        result = resp['result']
        schema = result.get('schema', {})
        fields = [get_property(field, 'name') for field in schema.get('fields', [])]
        data = result.get('data', [])

        data_dict = {field: [get_property(el, field) for el in data] for field in fields}
        display(pd.DataFrame(data_dict))

def handle_chart_response(resp):
    """Handles responses for generating charts."""
    if 'query' in resp:
        print(get_property(resp['query'], 'instructions'))
    elif 'result' in resp:
        vegaConfig = get_property(resp['result'], 'vegaConfig')
        if vegaConfig:
            alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()

def handle_error(resp):
    """Handles error responses."""
    display_section_title('Error')
    print(f"Code: {get_property(resp, 'code')}")
    print(f"Message: {get_property(resp, 'message')}")



In [ ]:

def get_stream(url, payload):
    """
    Posts data to a URL and processes the streaming JSON response line by line.

    Args:
        url (str): The URL to send the POST request to.
        payload (dict): The JSON payload to send in the request body.
    """
    s = requests.Session()
    acc = ''  # Accumulator for JSON parts

    try:
        with s.post(url, json=payload, headers=headers, stream=True) as resp:
            resp.raise_for_status()  # Raise an exception for bad status codes
            for line in resp.iter_lines():
                if not line:
                    continue

                try:
                    decoded_line = line.decode('utf-8')
                except UnicodeDecodeError:
                    print(f"Warning: Could not decode line: {line}")
                    continue

                # This custom JSON assembly logic seems fragile.
                # It's attempting to piece together a JSON object
                # from lines that might not be complete JSON objects themselves.
                if decoded_line == '[{':
                    acc = '{'
                elif decoded_line == '}]':
                    acc += '}'
                elif decoded_line == ',':
                    continue
                else:
                    acc += decoded_line

                if not is_json(acc):
                    continue

                try:
                    data_json = json_lib.loads(acc)
                except json_lib.JSONDecodeError:
                    print(f"Warning: Could not decode accumulated JSON: {acc}")
                    acc = '' # Reset accumulator on error
                    continue

                if 'error' in data_json:
                    handle_error(data_json['error'])
                    acc = ''
                    continue

                if 'systemMessage' in data_json:
                    system_message = data_json['systemMessage']
                    if 'text' in system_message:
                        handle_text_response(system_message['text'])
                    elif 'schema' in system_message:
                        handle_schema_response(system_message['schema'])
                    elif 'data' in system_message:
                        handle_data_response(system_message['data'])
                    elif 'chart' in system_message:
                        handle_chart_response(system_message['chart'])
                    else:
                        # Fallback for unhandled systemMessage types
                        colored_json = highlight(acc, lexers.JsonLexer(), formatters.TerminalFormatter())
                        print(colored_json)
                else:
                    # Fallback for responses without systemMessage or error
                    colored_json = highlight(acc, lexers.JsonLexer(), formatters.TerminalFormatter())
                    print(colored_json)

                acc = ''  # Reset accumulator after processing a complete JSON object

    except requests.exceptions.RequestException as e:
        print(f"Error during request: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")



In [ ]:
def get_stream_multi_turn(url, json, conversation_messages):
    s = requests.Session()

    acc = ''

    with s.post(url, json=json, headers=headers, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue

            decoded_line = str(line, encoding='utf-8')

            if decoded_line == '[{':
                acc = '{'
            elif decoded_line == '}]':
                acc += '}'
            elif decoded_line == ',':
                continue
            else:
                acc += decoded_line

            if not is_json(acc):
                continue

            data_json = json_lib.loads(acc)
            # Store the response that will be used in the next iteration
            conversation_messages.append(data_json)

            if not 'systemMessage' in data_json:
                if 'error' in data_json:
                    handle_error(data_json['error'])
                continue

            if 'text' in data_json['systemMessage']:
                handle_text_response(data_json['systemMessage']['text'])
            elif 'schema' in data_json['systemMessage']:
                handle_schema_response(data_json['systemMessage']['schema'])
            elif 'data' in data_json['systemMessage']:
                handle_data_response(data_json['systemMessage']['data'])
            elif 'chart' in data_json['systemMessage']:
                handle_chart_response(data_json['systemMessage']['chart'])
            else:
                colored_json = highlight(acc, lexers.JsonLexer(), formatters.TerminalFormatter())
                print(colored_json)
            print('\n')
            acc = ''

In [ ]:
# %% [markdown]
# # Spanner Data Agent End-to-End Example
#
# This notebook demonstrates how to create and interact with a Data Agent for Spanner using the Gemini Data Analytics API via HTTP requests in Python.

# %% [markdown]
# ## 1. Setup and Authentication
#
# Import necessary libraries and authenticate your user. This will be used to authorize the API requests.

# %% code
import json
import requests
from google.colab import auth
import os

# Authenticate user and get access token
auth.authenticate_user()
access_token = !gcloud auth application-default print-access-token

if not access_token or not access_token[0]:
    raise ValueError("Failed to get access token. Please ensure you are authenticated.")

headers = {
    "Authorization": f"Bearer {access_token[0]}",
    "Content-Type": "application/json",
    "x-server-timeout": "300",  # Custom timeout up to 600s
}
print("Headers configured.")

In [ ]:
# %% [markdown]
# ## 2. Configure Data Source Details
#
# Update the variables below with your specific Google Cloud project, Spanner instance details, and desired agent settings.

# %% code
# Replace with your actual project and Spanner details
billing_project = "project-id"  # Your billing project ID
location = "global"  # The region of your Spanner instance

# Spanner connection details
spanner_project_id = "project-id"  # Project ID of the Spanner instance
spanner_instance_id = "instance-id"  # Your spanner instance ID
spanner_database_id = "database-id"  # Your database name
engine = "GOOGLE_SQL"
system_instruction = "Help the user analyze data from the spanner database"

# Data Agent ID
data_agent_id = "spanner_agent_e2e_example"

In [ ]:
# %% [markdown]
# ## 3. Define Spanner Data Source
#
# This dictionary structure tells the Data Agent how to connect to your Spanner database.
# Its better to provide context_set_id similar to https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/data-agent-authored-context-databases#providing_context_with_querydata

# %% code
# Spanner data source definition
spanner_data_sources = {
    "spanner_reference": {
        "database_reference": {
            "project_id": spanner_project_id,
            "region": location,
            "engine": engine,
            "instance_id": spanner_instance_id,
            "database_id": spanner_database_id,
        },
        # Optional: Include this if you have pre-authored context for the agent
        # "agent_context_reference": {
        #     "context_set_id": f"projects/{billing_project}/locations/{location}/contextSets/your_context_set_id"
        # }
    }
}
print("Spanner data source configured:")
print(json.dumps(spanner_data_sources, indent=2))


In [ ]:
# %% [markdown]
# ## 4. Create the Data Agent
#
# Send a POST request to the Gemini Data Analytics API to create the Data Agent resource.
# Make sure roles/geminidataanalytics.dataAgentCreator is granted

# %% code
data_agent_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/global/dataAgents"

data_agent_payload = {
    "name": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
    "description": "This is an example Spanner data agent created from Colab.",  # Optional
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": spanner_data_sources,
            "system_instruction": system_instruction,
        }
    }
}

params = {"data_agent_id": data_agent_id}  # Optional

print(f"Creating Data Agent: {data_agent_id}...")
data_agent_response = requests.post(
    data_agent_url, params=params, json=data_agent_payload, headers=headers
)

if data_agent_response.status_code == 200:
    print("Data Agent created successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
elif data_agent_response.status_code == 409:
    print(f"Data Agent '{data_agent_id}' already exists.")
    # Optionally, you could add code here to fetch the existing agent
else:
    print(f"Error creating Data Agent: {data_agent_response.status_code}")
    print(data_agent_response.text)

In [ ]:
# %% [markdown]
# ## 5. Create the conversation
#
# Send a POST request to the Gemini Data Analytics API to create conversation.

# %% code
conversation_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/global/conversations"

conversation_id = "conversation_agent_spanner_example"

conversation_payload = {
    "agents": [
        f"projects/{billing_project}/locations/global/dataAgents/{data_agent_id}"
    ],
    "name": f"projects/{billing_project}/locations/global/conversations/{conversation_id}"
}

params = {
    "conversation_id": conversation_id
}

conversation_response = requests.post(conversation_url, headers=headers, params=params, json=conversation_payload)

if conversation_response.status_code == 200:
    print("Conversation created successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error creating Conversation: {conversation_response.status_code}")
    print(conversation_response.text)

In [ ]:
# %% [markdown]
# ## 6. Chat with the API by using conversation (stateful)
#
# Send a POST request to the Gemini Data Analytics API to chat using a conversation

# %% code

chat_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/global:chat"

# The natural language question to ask the data agent
user_prompt = "Which artist has the most cards in the database?"  # Replace with your question


# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [
        {
            "userMessage": {
                "text": user_prompt
            }
        }
    ],
    "conversation_reference": {
        "conversation": f"projects/{billing_project}/locations/global/conversations/{conversation_id}",
        "data_agent_context": {
            "data_agent": f"projects/{billing_project}/locations/global/dataAgents/{data_agent_id}",
        }
    }
}

print(f"Sending prompt to :chat: '{user_prompt}'")
print(f"Endpoint: {chat_url}")
print(f"Payload: {json.dumps(chat_payload, indent=2)}")


get_stream(chat_url, chat_payload)

In [ ]:
# %% [markdown]
# ## 7. Chat with the API by using data agent (stateless)
#
# Send a POST request to the Gemini Data Analytics API to chat without conversation
# %% code

chat_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/{location}:chat"


# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [
        {
            "userMessage": {
                "text": user_prompt
            }
        }
    ],
     "data_agent_context": {
          "data_agent": f"projects/{billing_project}/locations/global/dataAgents/{data_agent_id}",
     }
}

print(f"Sending prompt to :chat: '{user_prompt}'")
print(f"Endpoint: {chat_url}")
print(f"Payload: {json.dumps(chat_payload, indent=2)}")

get_stream(chat_url, chat_payload)


In [ ]:
# %% [markdown]
# ## 8. Chat with the API by using inline context (stateless)
#
# Send a POST request to the Gemini Data Analytics API to chat with inline context
# %% code

chat_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/us-east4:chat"


# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/{location}",
    "messages": [
        {
            "userMessage": {
                "text": user_prompt
            }
        }
    ],
     "inline_context": {
        "datasource_references": spanner_data_sources
        # The "options" field for python analysis is not shown in the Spanner example
    }
}

print(f"Sending prompt to :chat: '{user_prompt}'")
print(f"Endpoint: {chat_url}")
print(f"Payload: {json.dumps(chat_payload, indent=2)}")

get_stream(chat_url, chat_payload)


In [ ]:
# %% [markdown]
# ## 9. Chat with the API multi turn conversation
#
# Send a POST request to the Gemini Data Analytics API to chat with multi turn conversation
# %% code


chat_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{billing_project}/locations/{location}:chat"

# List that is used to track previous turns and is reused across requests
conversation_messages = []

# Helper function for calling the API
def multi_turn_conversation(msg):
    userMessage = {
        "userMessage": {
            "text": msg
        }
    }

    # Send a multi-turn request by including previous turns and the new message
    conversation_messages.append(userMessage)
    print(f"Current conversation history: {json.dumps(conversation_messages, indent=2)}")

    # Construct the payload for Spanner
    chat_payload = {
        "parent": f"projects/{billing_project}/locations/{location}",
        "messages": conversation_messages,
        "data_agent_context": {
              "data_agent": f"projects/{billing_project}/locations/global/dataAgents/{data_agent_id}",
              # "credentials": looker_credentials
          },

    }

    # Call the get_stream_multi_turn helper function to stream the response
    get_stream_multi_turn(chat_url, chat_payload, conversation_messages)

# Send first-turn request
print("--- Turn 1 ---")
multi_turn_conversation("Which artist has the most cards in the database?")

# Send follow-up-turn request
print("\n--- Turn 2 ---")
multi_turn_conversation("How many unique artists are represented in the database?")

In [ ]:

# %% [markdown]
# ## 6. Clean up (Optional)
#
# Delete the Data Agent resource if it's no longer needed.

# %% code
def delete_data_agent(project, agent_id):
    delete_url = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{project}/locations/global/dataAgents/{agent_id}"
    print(f"Deleting Data Agent: {agent_id}...")
    delete_response = requests.delete(delete_url, headers=headers)
    if delete_response.status_code == 200:
        print(f"Data Agent {agent_id} deleted successfully.")
    elif delete_response.status_code == 404:
        print(f"Data Agent {agent_id} not found.")
    else:
        print(f"Error deleting Data Agent {agent_id}: {delete_response.status_code}")
        print(delete_response.text)

# Uncomment to delete the agent:
delete_data_agent(billing_project, data_agent_id)

